# Inspect Margulis WUS-SR (NSIDC-0719) raw granules

Characterise one raw NSIDC-0719 granule before building the per-calendar-year
CF NetCDF consolidator (issue #111, template PR #109).

**Questions this notebook answers** — each informs the consolidator design:

- What is the on-disk variable name for SWE? (catalog declares `SWE`; granules
  use `SWE_Post`)
- What does the `Stats` dimension represent, and which index is the posterior
  mean?
- What are the coordinate names and ordering conventions? (capital-L `Latitude`
  / `Longitude`, descending latitude)
- How long is the `Day` axis, and does it line up with a 365/366-day water
  year starting Oct 1?
- How are tiles named, and do adjacent tiles share a regular grid that can
  be merged with `combine='by_coords'`?
- Does the calendar-year rebuild (Jan–Sep of WY *X* + Oct–Dec of WY *X+1*)
  yield exactly 365 / 366 daily steps for the target calendar year?

The consolidator in `fetch/margulis_wus_sr.py` codifies the answers; this
notebook is the source of truth for **why** those decisions were made.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# --- Configuration ---
DATASTORE = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore"
)
RAW_ROOT = DATASTORE / "margulis_wus_sr" / "raw"
CY = 2000  # calendar year to characterise
TILE = "N31_0W103_0"  # one tile of the WUS domain
VERSION = "v01"
AGG = "agg_16"  # 16x coarsening from the native 90 m product


def granule_path(cy: int, tile: str, wy_prev: int) -> Path:
    """Build the path to one tile-WY granule under raw/<cy>/."""
    wy_curr = (wy_prev + 1) % 100
    return (
        RAW_ROOT
        / f"{cy}"
        / (
            f"WUS_UCLA_SR_{VERSION}_{tile}_{AGG}_"
            f"WY{wy_prev}_{wy_curr:02d}_SWE_SCA_POST.nc"
        )
    )


# CY 2000 is assembled from WY1984_85... wait, CY 2000 from WY1999_00 + WY2000_01.
# WY1999_00 = Oct 1999 – Sep 2000  -> Jan–Sep 2000
# WY2000_01 = Oct 2000 – Sep 2001  -> Oct–Dec 2000
WY_PREV_FIRST = 1999  # the WY that *ends* in Sep of CY 2000
WY_PREV_SECOND = 2000  # the WY that *starts* in Oct of CY 2000

p1 = granule_path(CY, TILE, WY_PREV_FIRST)
p2 = granule_path(CY, TILE, WY_PREV_SECOND)
print("WY ending in Sep 2000 :", p1.name, "  exists:", p1.exists())
print("WY starting Oct 2000  :", p2.name, "  exists:", p2.exists())

## 1. Granule schema — dims, vars, units, attrs

Confirm the on-disk variable name (`SWE_Post`), the units string (`meters`),
and the dimension order. The catalog declares variable name `SWE` and
`cf_units: m`, so the consolidator will rename `SWE_Post` → `SWE`.

In [ ]:
ds = xr.open_dataset(p1)
print(ds)
print()
for v in ds.data_vars:
    print(f"  {v}: dims={ds[v].dims}, attrs={dict(ds[v].attrs)}")
for c in ds.coords:
    print(
        f"  {c}: shape={ds[c].shape}, attrs={dict(ds[c].attrs)},"
        f" first 3 = {ds[c].values[:3].tolist()},"
        f" last = {ds[c].values[-1]}"
    )

## 2. `Stats` dimension semantics

NSIDC-0719 stores 5 posterior statistics along the `Stats` dimension. The
dataset README documents these as (in order):

1. **mean** (posterior expectation, in metres SWE)
2. **standard deviation**
3. **median**
4. **25th percentile**
5. **75th percentile**

We can sanity-check by picking a snow-season day and inspecting the per-pixel
values across `Stats` for a single grid cell. The mean and median should be
of similar magnitude; the std should be small and non-negative; the 25th and
75th percentiles should bracket the median.

In [ ]:
# Pick April 1 of CY 2000 -> day_of_wy = 31+30+31 (OND) + 31+29+31 (JFM) + 1 = 183
# (1-indexed). WY1999_00 day index 182 (0-indexed) should be Apr 1, 2000.
day_of_wy_0idx = 182

ix_lat = ds["Latitude"].size // 2
ix_lon = ds["Longitude"].size // 2
swe_stats = ds["SWE_Post"].isel(Day=day_of_wy_0idx, Latitude=ix_lat, Longitude=ix_lon)
print("5 Stats values at one pixel on Apr 1, 2000 (m SWE):")
for k, v in enumerate(swe_stats.values):
    print(f"  Stats[{k}] = {float(v):.4f}")
print(
    "\nExpected ordering: [mean, std, median, p25, p75]. "
    "Std should be smallest and non-negative; "
    "p25 < median ≈ mean < p75."
)

## 3. Spatial-mean SWE seasonal cycle (one tile)

Plot the tile-averaged posterior mean SWE across the water year. Expect:

- Near-zero late summer / fall
- Build-up Dec–Mar
- Peak Mar–Apr
- Ablation Apr–Jun

In [ ]:
# WY1999_00 has 366 days (CY 2000 is a leap year). Day 1 = Oct 1, 1999.
wy_start = pd.Timestamp(f"{WY_PREV_FIRST}-10-01")
ndays = ds["Day"].size
time = pd.date_range(wy_start, periods=ndays, freq="D")
print(
    f"WY1999_00 day count: {ndays} (expect 366 since CY 2000 is leap year)\n"
    f"First day: {time[0].date()}, last day: {time[-1].date()}"
)

swe_mean = ds["SWE_Post"].isel(Stats=0).mean(dim=["Latitude", "Longitude"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time, swe_mean.values, color="steelblue")
ax.set_ylabel("Tile-mean SWE (m)")
ax.set_title(
    f"Margulis WUS-SR posterior mean SWE — tile {TILE} — WY{WY_PREV_FIRST+1}"
)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Calendar-year rebuild check

The consolidator builds calendar year *X* from:

- Jan–Sep of *X* from WY ending in Sep of *X* (days 93–end)
- Oct–Dec of *X* from WY starting Oct of *X* (days 1–92)

Open both WYs, slice each, concat along time, and verify the time axis is
exactly 365 (or 366 for leap years) days covering Jan 1 – Dec 31 of CY.

In [ ]:
def _wy_to_time(ds: xr.Dataset, wy_prev_year: int) -> xr.Dataset:
    """Replace 0-indexed `Day` with daily timestamps starting Oct 1 of wy_prev_year."""
    wy_start = pd.Timestamp(f"{wy_prev_year}-10-01")
    times = pd.date_range(wy_start, periods=ds["Day"].size, freq="D")
    return ds.assign_coords(Day=times).rename({"Day": "time"})


ds1 = _wy_to_time(xr.open_dataset(p1), WY_PREV_FIRST)
ds2 = _wy_to_time(xr.open_dataset(p2), WY_PREV_SECOND)

cy_start = pd.Timestamp(f"{CY}-01-01")
cy_end = pd.Timestamp(f"{CY}-12-31")
ds1_cy = ds1.sel(time=slice(cy_start, cy_end))
ds2_cy = ds2.sel(time=slice(cy_start, cy_end))
ds_cy = xr.concat([ds1_cy, ds2_cy], dim="time").sortby("time")

print(f"CY {CY} time range: {ds_cy.time.values[0]} → {ds_cy.time.values[-1]}")
print(f"CY {CY} day count : {ds_cy.time.size}  (expect 366 for leap year 2000)")
assert ds_cy.time.size == 366, "Expected 366 days in CY 2000"
assert pd.Timestamp(ds_cy.time.values[0]) == cy_start
assert pd.Timestamp(ds_cy.time.values[-1]) == cy_end
print("✓ boundary logic produces a complete calendar year")

## 5. Tile geometry inventory

List all tile bbox extents in `raw/<CY>/`. Adjacent tiles should share grid
edges (one tile's max lat == next tile's min lat) so `xr.combine_by_coords`
can mosaic them without resampling.

In [ ]:
tile_files = sorted((RAW_ROOT / f"{CY}").glob(f"*WY{WY_PREV_FIRST}_*_SWE_SCA_POST.nc"))
print(f"Number of tiles in WY{WY_PREV_FIRST + 1}: {len(tile_files)}")

extents = []
for f in tile_files[:8]:  # sample
    with xr.open_dataset(f) as t:
        extents.append(
            {
                "tile": f.name.split("_v01_")[1].split("_agg_")[0],
                "lat_min": float(t["Latitude"].min()),
                "lat_max": float(t["Latitude"].max()),
                "lon_min": float(t["Longitude"].min()),
                "lon_max": float(t["Longitude"].max()),
                "nlat": t["Latitude"].size,
                "nlon": t["Longitude"].size,
            }
        )
print(pd.DataFrame(extents).to_string(index=False))

## 6. Spot-check the consolidator output

Once `consolidate_calendar_year_margulis_wus_sr` is implemented, re-open one
of its outputs here and confirm:

- variable named `SWE` (not `SWE_Post`)
- `time`, `lat`, `lon` coords (not `Day`/`Latitude`/`Longitude`)
- `Conventions = CF-1.6` global attr
- `crs` ancillary variable with WGS84 grid mapping
- 365 or 366 daily steps Jan 1 – Dec 31 of the target year

In [ ]:
DAILY_DIR = DATASTORE / "margulis_wus_sr" / "daily"
consolidated = DAILY_DIR / f"margulis_wus_sr_daily_{CY}.nc"
if consolidated.exists():
    ds_out = xr.open_dataset(consolidated)
    print(ds_out)
    print("\nConventions:", ds_out.attrs.get("Conventions"))
    print("crs var present:", "crs" in ds_out.variables)
    print("time size:", ds_out.dims.get("time"))
else:
    print(
        f"Consolidated file not yet built at {consolidated}; "
        "run `nhf-targets fetch margulis-wus-sr --period {CY}/{CY+1}` "
        "or call `consolidate_calendar_year_margulis_wus_sr` directly."
    )